# Kalman Filter Assignment — Solutions

This notebook contains full analytical derivations and Python implementations for all three questions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib import rc
from scipy.stats import norm
from IPython.display import HTML

np.random.seed(42)
print("Libraries loaded.")

---
# Q1. Analytical Derivation

## Filter Model

$$x^{-}_k = A_{k-1}\,x^{+}_{k-1} + G_{k-1}\,w_{k-1}, \qquad w_{k-1} \sim \mathscr{N}(0, \Sigma_p)$$
$$y^{-}_k = H_k\,x^{-}_k + z_k, \qquad z_k \sim \mathscr{N}(0, \Sigma_m)$$
$$x^{+}_{k-1} \sim \mathscr{N}(m_{k-1}, P_{k-1})$$

All noise terms and the initial state are mutually independent.

---

## Part 1: Distribution of $x_k^-$

**Claim:** $x_k^- \sim \mathscr{N}(m_k^-, P_k^-)$ where $m_k^- = A_{k-1}m_{k-1}$ and $P_k^- = A_{k-1}P_{k-1}A_{k-1}^T + G_{k-1}\Sigma_p G_{k-1}^T$.

**Proof:**

Since $x^+_{k-1} \sim \mathscr{N}(m_{k-1}, P_{k-1})$ and $w_{k-1} \sim \mathscr{N}(0, \Sigma_p)$ are independent, and $x_k^- = A_{k-1}x^+_{k-1} + G_{k-1}w_{k-1}$ is an affine transformation of jointly Gaussian random variables, $x_k^-$ is also Gaussian.

**Mean:**
$$\mathbb{E}[x_k^-] = A_{k-1}\,\mathbb{E}[x^+_{k-1}] + G_{k-1}\,\mathbb{E}[w_{k-1}] = A_{k-1}m_{k-1} + G_{k-1}\cdot 0 = m_k^-$$

**Covariance:** Let $\delta x = x^+_{k-1} - m_{k-1}$. Then:
$$x_k^- - m_k^- = A_{k-1}\delta x + G_{k-1}w_{k-1}$$

$$P_k^- = \mathbb{E}[(x_k^- - m_k^-)(x_k^- - m_k^-)^T]$$
$$= A_{k-1}\,\mathbb{E}[\delta x\,\delta x^T]A_{k-1}^T + G_{k-1}\,\mathbb{E}[w_{k-1}w_{k-1}^T]G_{k-1}^T + \text{cross terms}$$

Cross terms vanish by independence of $w_{k-1}$ and $x^+_{k-1}$. Therefore:
$$\boxed{P_k^- = A_{k-1}P_{k-1}A_{k-1}^T + G_{k-1}\Sigma_p G_{k-1}^T}$$

---

## Part 2: Distribution of $y_k^-$

**Claim:** $y_k^- \sim \mathscr{N}(H_k m_k^-,\; H_k P_k^- H_k^T + \Sigma_m)$

**Proof:**

Since $y_k^- = H_k x_k^- + z_k$ is an affine transformation of Gaussian $x_k^-$ and independent Gaussian $z_k$:

**Mean:**
$$\mathbb{E}[y_k^-] = H_k\,\mathbb{E}[x_k^-] + \mathbb{E}[z_k] = H_k m_k^- + 0 = H_k m_k^-$$

**Covariance:**
$$\text{Var}(y_k^-) = H_k\,\text{Var}(x_k^-)\,H_k^T + \text{Var}(z_k) = \boxed{H_k P_k^- H_k^T + \Sigma_m}$$

---

## Part 3: Joint Distribution of $(x_k^-, y_k^-)$

**Claim:** $\begin{bmatrix} x_k^- \\ y^{-}_k \end{bmatrix} \sim \mathscr{N}\left(\begin{bmatrix} m_k^- \\ H_k m_k^- \end{bmatrix}, \begin{bmatrix} P_k^- & P_k^- H_k^T \\ H_k P_k^- & H_k P_k^- H_k^T + \Sigma_m \end{bmatrix}\right)$

**Proof:**

The means follow directly from Parts 1 and 2. For the covariance matrix, we need the cross-covariance:

$$\text{Cov}(x_k^-, y_k^-) = \mathbb{E}[(x_k^- - m_k^-)(y_k^- - H_k m_k^-)^T]$$
$$= \mathbb{E}[(x_k^- - m_k^-)(H_k(x_k^- - m_k^-) + z_k)^T]$$
$$= \mathbb{E}[(x_k^- - m_k^-)(x_k^- - m_k^-)^T]H_k^T + \mathbb{E}[(x_k^- - m_k^-)z_k^T]$$
$$= P_k^- H_k^T + 0 = P_k^- H_k^T$$

where the cross term vanishes because $z_k$ is independent of $x_k^-$. The full joint covariance is therefore:
$$\Sigma = \begin{bmatrix} P_k^- & P_k^- H_k^T \\ H_k P_k^- & H_k P_k^- H_k^T + \Sigma_m \end{bmatrix}$$

---

## Part 4: Posterior $x_k^+ = (x_k^- \mid y_k^- = y_k^{\text{obs}})$

**Proof using the Gaussian conditioning formula:**

For a joint Gaussian $\begin{bmatrix} u \\ v \end{bmatrix} \sim \mathscr{N}\left(\begin{bmatrix}\mu_u \\ \mu_v\end{bmatrix}, \begin{bmatrix}\Sigma_{uu} & \Sigma_{uv} \\ \Sigma_{vu} & \Sigma_{vv}\end{bmatrix}\right)$, the conditional is:
$$u \mid v = v_0 \sim \mathscr{N}(\mu_u + \Sigma_{uv}\Sigma_{vv}^{-1}(v_0 - \mu_v),\; \Sigma_{uu} - \Sigma_{uv}\Sigma_{vv}^{-1}\Sigma_{vu})$$

Applying this with $u = x_k^-$, $v = y_k^-$, $v_0 = y_k^{\text{obs}}$:

- $\Sigma_{uv} = P_k^- H_k^T$
- $\Sigma_{vv} = H_k P_k^- H_k^T + \Sigma_m$
- Define $K_k \triangleq \Sigma_{uv}\Sigma_{vv}^{-1} = P_k^- H_k^T(H_k P_k^- H_k^T + \Sigma_m)^{-1}$ (Kalman Gain)

**Updated mean:**
$$m_k = m_k^- + K_k(y_k^{\text{obs}} - H_k m_k^-)$$

**Updated covariance:**
$$P_k = P_k^- - P_k^- H_k^T(H_k P_k^- H_k^T + \Sigma_m)^{-1}H_k P_k^-$$
$$= P_k^- - K_k H_k P_k^- = (I - K_k H_k)P_k^- \qquad \blacksquare$$

---

## Part 5: Conditional Mean and Variance of $x_k^-$ given $y_k^- = y_k^{\text{obs}}$

From Part 4 (the posterior IS the conditional distribution $x_k^- \mid y_k^- = y_k^{\text{obs}}$):

$$\boxed{\mathbb{E}[x_k^- \mid y_k^- = y_k^{\text{obs}}] = m_k = m_k^- + K_k(y_k^{\text{obs}} - H_k m_k^-)}$$

$$\boxed{\text{Var}(x_k^- \mid y_k^- = y_k^{\text{obs}}) = P_k = (I - K_k H_k)P_k^-}$$

Note: $P_k \preceq P_k^-$ (the posterior covariance is always less than or equal to the prior covariance), confirming that the measurement always reduces uncertainty.

---
# Q2. 1-D Example

## Scalar model:
$$x^-_k = a\,x^+_{k-1} + w_{k-1}, \quad w_{k-1}\sim\mathscr{N}(0,q)$$
$$y^-_k = h\,x^-_k + z_k, \quad z_k\sim\mathscr{N}(0,r)$$

---

## Part 1: Prediction step (scalar case)

Setting $A_{k-1} = a$, $G_{k-1} = 1$, $\Sigma_p = q$ in Q1 Part 1:

$$\boxed{m_k^- = a\,m_{k-1}}$$
$$\boxed{P_k^- = a^2 P_{k-1} + q}$$

---

## Part 2: Update step (scalar case)

With $H_k = h$, $\Sigma_m = r$, the innovation variance is $S_k = h^2 P_k^- + r$.

The scalar Kalman gain is:
$$K_k = \frac{P_k^- h}{S_k} = \frac{P_k^- h}{h^2 P_k^- + r}$$

Updated mean:
$$\boxed{m_k = m_k^- + K_k v_k = m_k^- + \frac{P_k^- h}{S_k}(y_k^{\text{obs}} - h\,m_k^-)}$$

Updated variance:
$$\boxed{P_k = (1 - K_k h)P_k^- = \left(1 - \frac{P_k^- h^2}{S_k}\right)P_k^-}$$

---

## Part 3: Predictive measurement distribution

Before seeing $y_k$, from Q1 Part 2 with scalar parameters:
$$\boxed{p(y_k^- \mid Y_{k-1}) = \mathscr{N}(h\,m_k^-,\; h^2 P_k^- + r) = \mathscr{N}(h\,m_k^-,\; S_k)}$$

---

## Part 4: Posterior-predictive measurement distribution

After assimilating $y_k$, the state is $x_k^+ \sim \mathscr{N}(m_k, P_k)$. A future replicate measurement $y_k^{\text{rep}} = h x_k^+ + z$ with $z \sim \mathscr{N}(0,r)$ independent gives:
$$\boxed{p(y_k^- \mid Y_k) = \mathscr{N}(h\,m_k,\; h^2 P_k + r)}$$

---

## Part 5: Animation — Prior vs Posterior

In [ ]:
# ── 1-D Kalman Filter: scalar parameters ──────────────────────────────────────
a = 0.95     # state transition
q = 1.0      # process noise variance
h = 1.0      # observation matrix
r = 2.0      # measurement noise variance

# Prior
m0, P0 = 0.0, 5.0

# True state trajectory
T = 20
x_true = np.zeros(T)
x_true[0] = np.random.normal(m0, np.sqrt(P0))
for k in range(1, T):
    x_true[k] = a * x_true[k-1] + np.random.normal(0, np.sqrt(q))

y_obs = h * x_true + np.random.normal(0, np.sqrt(r), T)

# Run scalar Kalman filter
m_pred = np.zeros(T)
P_pred = np.zeros(T)
m_filt = np.zeros(T)
P_filt = np.zeros(T)

m_prev, P_prev = m0, P0
for k in range(T):
    # Predict
    mp = a * m_prev
    Pp = a**2 * P_prev + q
    m_pred[k], P_pred[k] = mp, Pp
    # Update
    S = h**2 * Pp + r
    K = Pp * h / S
    mf = mp + K * (y_obs[k] - h * mp)
    Pf = (1 - K * h) * Pp
    m_filt[k], P_filt[k] = mf, Pf
    m_prev, P_prev = mf, Pf

print(f"Filter ran for {T} steps.")
print(f"Final filtered mean: {m_filt[-1]:.3f}, variance: {P_filt[-1]:.3f}")

In [ ]:
# ── Static summary plot ────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

steps = np.arange(T)
ax = axes[0]
ax.plot(steps, x_true, 'k-o', ms=4, label='True state')
ax.plot(steps, y_obs, 'rx', ms=6, label='Noisy measurements')
ax.plot(steps, m_pred, 'b--', label='Predicted mean $m_k^-$')
ax.plot(steps, m_filt, 'g-', label='Filtered mean $m_k$')
ax.fill_between(steps,
                m_filt - 2*np.sqrt(P_filt),
                m_filt + 2*np.sqrt(P_filt),
                alpha=0.2, color='green', label='±2σ posterior')
ax.set_xlabel('Time step k')
ax.set_ylabel('State')
ax.set_title('1-D Kalman Filter: State Estimates')
ax.legend()
ax.grid(True, alpha=0.3)

ax2 = axes[1]
ax2.plot(steps, P_pred, 'b--o', ms=4, label='Prior variance $P_k^-$')
ax2.plot(steps, P_filt, 'g-o', ms=4, label='Posterior variance $P_k$')
ax2.set_xlabel('Time step k')
ax2.set_ylabel('Variance')
ax2.set_title('Prior vs Posterior Variance Over Time')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('1d_kalman_summary.png', dpi=120, bbox_inches='tight')
plt.show()
print("Summary plot saved.")

In [ ]:
# ── Animation: Prior vs Posterior Gaussian distributions ──────────────────────
rc('animation', html='jshtml')

fig, ax = plt.subplots(figsize=(10, 5))

x_range = np.linspace(-15, 15, 500)

line_prior, = ax.plot([], [], 'b--', lw=2, label='Prior $p(x_k^- | Y_{k-1})$')
line_post,  = ax.plot([], [], 'g-',  lw=2, label='Posterior $p(x_k^+ | Y_k)$')
line_meas,  = ax.plot([], [], 'r--', lw=1.5, label='Meas. likelihood (unnorm.)')
obs_line    = ax.axvline(x=0, color='red', lw=1.5, linestyle=':', label='Observation')
true_line   = ax.axvline(x=0, color='black', lw=1.5, linestyle='-', label='True state')

ax.set_xlim(x_range[0], x_range[-1])
ax.set_ylim(0, 0.8)
ax.set_xlabel('State x')
ax.set_ylabel('Density')
title = ax.set_title('')
ax.legend(loc='upper right', fontsize=8)
ax.grid(True, alpha=0.3)

def init():
    line_prior.set_data([], [])
    line_post.set_data([], [])
    line_meas.set_data([], [])
    return line_prior, line_post, line_meas, obs_line, true_line, title

def animate(k):
    # Prior distribution: N(m_pred[k], P_pred[k])
    prior_pdf = norm.pdf(x_range, m_pred[k], np.sqrt(P_pred[k]))
    # Posterior distribution: N(m_filt[k], P_filt[k])
    post_pdf  = norm.pdf(x_range, m_filt[k], np.sqrt(P_filt[k]))
    # Likelihood: N(y_obs[k]; h*x, r)
    like_pdf  = norm.pdf(x_range, y_obs[k]/h, np.sqrt(r)/abs(h))
    like_pdf  = like_pdf / (like_pdf.max() + 1e-10) * post_pdf.max() * 0.8

    line_prior.set_data(x_range, prior_pdf)
    line_post.set_data(x_range, post_pdf)
    line_meas.set_data(x_range, like_pdf)
    obs_line.set_xdata([y_obs[k], y_obs[k]])
    true_line.set_xdata([x_true[k], x_true[k]])
    title.set_text(f'Step k={k}  |  True={x_true[k]:.2f},  Obs={y_obs[k]:.2f},  '
                   f'Prior mean={m_pred[k]:.2f},  Post mean={m_filt[k]:.2f}')

    # Dynamic y-axis
    ymax = max(prior_pdf.max(), post_pdf.max()) * 1.3
    ax.set_ylim(0, max(ymax, 0.05))
    return line_prior, line_post, line_meas, obs_line, true_line, title

ani = animation.FuncAnimation(fig, animate, frames=T, init_func=init,
                               interval=800, blit=False)
plt.close()
ani

---
# Q3. 2D Position Estimation

## Part A: Derivation of System Matrices

### State and Constant-Velocity Kinematics

The state is $x_k = [p_x(k),\; p_y(k),\; v_x(k),\; v_y(k)]^T$.

Under constant-velocity motion with time step $\Delta t$, the kinematic equations are:
$$p_x(k) = p_x(k-1) + \Delta t\,v_x(k-1) + \tfrac{1}{2}\Delta t^2\, a_x$$
$$p_y(k) = p_y(k-1) + \Delta t\,v_y(k-1) + \tfrac{1}{2}\Delta t^2\, a_y$$
$$v_x(k) = v_x(k-1) + \Delta t\, a_x$$
$$v_y(k) = v_y(k-1) + \Delta t\, a_y$$

where accelerations $[a_x, a_y]^T = w_{k-1} \sim \mathscr{N}(0, \Sigma_p)$ are process noise.

Writing in matrix form $x_k^- = A x_{k-1}^+ + G w_{k-1}$:

$$A = \begin{bmatrix}1 & 0 & \Delta t & 0\\0 & 1 & 0 & \Delta t\\0 & 0 & 1 & 0\\0 & 0 & 0 & 1\end{bmatrix}$$

The noise input matrix maps $[a_x, a_y]^T$ to state increments:
$$G = \begin{bmatrix}\frac{1}{2}\Delta t^2 & 0\\0 & \frac{1}{2}\Delta t^2\\\Delta t & 0\\0 & \Delta t\end{bmatrix}$$

The measurement is only the position:
$$y_k = \begin{bmatrix}p_x(k)\\p_y(k)\end{bmatrix} = \begin{bmatrix}1 & 0 & 0 & 0\\0 & 1 & 0 & 0\end{bmatrix}x_k + z_k$$

$$H = \begin{bmatrix}1 & 0 & 0 & 0\\0 & 1 & 0 & 0\end{bmatrix} \qquad \blacksquare$$

---

## Part B: Python Implementation — 2D Kalman Filter for GPS

In [ ]:
# ── 2D Kalman Filter Implementation ───────────────────────────────────────────

def build_system_matrices(dt, sigma_a=0.5, sigma_m=2.0):
    """
    Build state-space matrices for 2D constant-velocity model.

    Args:
        dt      : time step (seconds)
        sigma_a : std dev of acceleration process noise (m/s^2)
        sigma_m : std dev of measurement noise (meters)
    Returns:
        A, G, H, Sigma_p, Sigma_m
    """
    A = np.array([[1, 0, dt, 0],
                  [0, 1, 0,  dt],
                  [0, 0, 1,  0],
                  [0, 0, 0,  1]], dtype=float)

    G = np.array([[0.5*dt**2, 0],
                  [0, 0.5*dt**2],
                  [dt, 0],
                  [0, dt]], dtype=float)

    H = np.array([[1, 0, 0, 0],
                  [0, 1, 0, 0]], dtype=float)

    Sigma_p = sigma_a**2 * np.eye(2)
    Sigma_m = sigma_m**2 * np.eye(2)

    return A, G, H, Sigma_p, Sigma_m


def kalman_filter_2d(y_obs_seq, A, G, H, Sigma_p, Sigma_m, m0, P0):
    """
    Run the Kalman filter over a sequence of 2D position measurements.

    Args:
        y_obs_seq : (T, 2) array of noisy GPS measurements
        A, G, H   : system matrices
        Sigma_p   : process noise covariance (2x2)
        Sigma_m   : measurement noise covariance (2x2)
        m0        : initial state mean (4,)
        P0        : initial state covariance (4x4)

    Returns:
        m_pred, P_pred : predicted (prior) means/covariances
        m_filt, P_filt : filtered (posterior) means/covariances
        K_gains        : Kalman gains at each step
    """
    T = len(y_obs_seq)
    n = len(m0)   # state dim
    ny = H.shape[0]  # measurement dim

    m_pred = np.zeros((T, n))
    P_pred = np.zeros((T, n, n))
    m_filt = np.zeros((T, n))
    P_filt = np.zeros((T, n, n))
    K_gains = np.zeros((T, n, ny))

    m, P = m0.copy(), P0.copy()

    for k in range(T):
        # ── Prediction step ──────────────────────────────────
        mp = A @ m
        Pp = A @ P @ A.T + G @ Sigma_p @ G.T
        m_pred[k] = mp
        P_pred[k] = Pp

        # ── Update step ───────────────────────────────────────
        S = H @ Pp @ H.T + Sigma_m          # innovation covariance
        K = Pp @ H.T @ np.linalg.inv(S)     # Kalman gain
        v = y_obs_seq[k] - H @ mp           # innovation
        mf = mp + K @ v
        Pf = (np.eye(n) - K @ H) @ Pp

        m_filt[k] = mf
        P_filt[k] = Pf
        K_gains[k] = K
        m, P = mf, Pf

    return m_pred, P_pred, m_filt, P_filt, K_gains


print("2D Kalman filter functions defined.")

In [ ]:
# ── Simulate a GPS trajectory ──────────────────────────────────────────────────
np.random.seed(0)

dt       = 1.0     # seconds
T        = 80      # number of time steps
sigma_a  = 0.3     # process noise (m/s^2)
sigma_m  = 5.0     # GPS measurement noise (meters)

A, G, H, Sigma_p, Sigma_m_mat = build_system_matrices(dt, sigma_a, sigma_m)

# True trajectory (figure-8 / curved path via random accelerations)
x_true_2d = np.zeros((T, 4))   # [px, py, vx, vy]
x_true_2d[0] = [0.0, 0.0, 5.0, 0.0]  # start at origin, moving east at 5 m/s

for k in range(1, T):
    w = np.random.multivariate_normal([0, 0], sigma_a**2 * np.eye(2))
    x_true_2d[k] = A @ x_true_2d[k-1] + G @ w

# Noisy GPS measurements (position only)
y_obs_2d = (H @ x_true_2d.T).T + np.random.multivariate_normal(
    [0, 0], sigma_m**2 * np.eye(2), T)

# Initial state estimate
m0 = np.array([y_obs_2d[0, 0], y_obs_2d[0, 1], 0.0, 0.0])
P0 = np.diag([sigma_m**2, sigma_m**2, 10.0**2, 10.0**2])

# Run filter
m_pred2, P_pred2, m_filt2, P_filt2, K_gains2 = kalman_filter_2d(
    y_obs_2d, A, G, H, Sigma_p, Sigma_m_mat, m0, P0)

print(f"Simulation and filtering complete. T={T} steps.")
print(f"RMS position error (GPS):    {np.sqrt(np.mean((y_obs_2d - (H @ x_true_2d.T).T)**2)):.2f} m")
print(f"RMS position error (Kalman): {np.sqrt(np.mean((m_filt2[:,:2] - x_true_2d[:,:2])**2)):.2f} m")

In [ ]:
# ── Plot 1: 2D trajectory comparison ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

ax = axes[0]
ax.plot(x_true_2d[:, 0], x_true_2d[:, 1], 'k-',  lw=2,  label='True trajectory')
ax.scatter(y_obs_2d[:, 0], y_obs_2d[:, 1], c='red', s=15, alpha=0.5, label=f'GPS (σ={sigma_m}m)')
ax.plot(m_filt2[:, 0], m_filt2[:, 1], 'g-', lw=2,  label='Kalman estimate')
ax.plot(m_filt2[0, 0], m_filt2[0, 1], 'go', ms=8)
ax.plot(x_true_2d[0, 0], x_true_2d[0, 1], 'k*', ms=12, label='Start')
ax.set_xlabel('x position (m)')
ax.set_ylabel('y position (m)')
ax.set_title('2D Position: True vs GPS vs Kalman Filter')
ax.legend()
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

ax2 = axes[1]
time = np.arange(T) * dt
err_gps    = np.sqrt(np.sum((y_obs_2d - x_true_2d[:, :2])**2, axis=1))
err_kalman = np.sqrt(np.sum((m_filt2[:, :2] - x_true_2d[:, :2])**2, axis=1))
ax2.plot(time, err_gps,    'r-', alpha=0.7, label='GPS error')
ax2.plot(time, err_kalman, 'g-', lw=2,      label='Kalman error')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Position error (m)')
ax2.set_title('Position Error Over Time')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('2d_kalman_trajectory.png', dpi=120, bbox_inches='tight')
plt.show()
print("Trajectory plot saved.")

In [ ]:
# ── Plot 2: Velocity estimation & uncertainty ──────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

time = np.arange(T) * dt

for i, (label, comp, col) in enumerate([('x', 0, 'b'), ('y', 1, 'r')]):
    # Position
    ax = axes[0, i]
    ax.plot(time, x_true_2d[:, comp], 'k-', lw=1.5, label=f'True p_{label}')
    ax.scatter(time, y_obs_2d[:, comp], c='gray', s=8, alpha=0.4, label='GPS')
    ax.plot(time, m_filt2[:, comp], f'{col}-', lw=2, label=f'Filtered $p_{label}$')
    std = np.sqrt(P_filt2[:, comp, comp])
    ax.fill_between(time, m_filt2[:, comp]-2*std, m_filt2[:, comp]+2*std,
                    alpha=0.2, color=col, label='±2σ')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel(f'$p_{label}$ (m)')
    ax.set_title(f'{label}-Position Filtering')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Velocity (latent — not measured)
    ax = axes[1, i]
    ax.plot(time, x_true_2d[:, comp+2], 'k-', lw=1.5, label=f'True $v_{label}$')
    ax.plot(time, m_filt2[:, comp+2], f'{col}-', lw=2, label=f'Estimated $v_{label}$')
    std_v = np.sqrt(P_filt2[:, comp+2, comp+2])
    ax.fill_between(time, m_filt2[:, comp+2]-2*std_v, m_filt2[:, comp+2]+2*std_v,
                    alpha=0.2, color=col, label='±2σ')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel(f'$v_{label}$ (m/s)')
    ax.set_title(f'{label}-Velocity Estimation (not measured!)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('2D Kalman Filter: Position & Velocity Components', fontsize=13)
plt.tight_layout()
plt.savefig('2d_kalman_components.png', dpi=120, bbox_inches='tight')
plt.show()
print("Component plot saved.")

In [ ]:
# ── Plot 3: Uncertainty ellipses at selected steps ─────────────────────────────
from matplotlib.patches import Ellipse

def covariance_ellipse(mean, cov2d, n_std=2.0, **kwargs):
    """Return a matplotlib Ellipse patch for a 2D Gaussian."""
    eigvals, eigvecs = np.linalg.eigh(cov2d)
    eigvals = np.maximum(eigvals, 0)
    angle = np.degrees(np.arctan2(eigvecs[1, 0], eigvecs[0, 0]))
    w, h = 2 * n_std * np.sqrt(eigvals)
    return Ellipse(xy=mean, width=w, height=h, angle=angle, **kwargs)

fig, ax = plt.subplots(figsize=(10, 8))

ax.plot(x_true_2d[:, 0], x_true_2d[:, 1], 'k-', lw=1.5, label='True trajectory', zorder=3)
ax.scatter(y_obs_2d[:, 0], y_obs_2d[:, 1], c='red', s=12, alpha=0.4, label='GPS obs', zorder=2)
ax.plot(m_filt2[:, 0], m_filt2[:, 1], 'g-', lw=2, label='Kalman estimate', zorder=4)

# Draw uncertainty ellipses at every 10th step
for k in range(0, T, 10):
    cov2d = P_filt2[k, :2, :2]
    ell = covariance_ellipse(m_filt2[k, :2], cov2d, n_std=2,
                              edgecolor='green', facecolor='green',
                              alpha=0.15, linewidth=1.5)
    ax.add_patch(ell)
    ax.annotate(f'k={k}', m_filt2[k, :2], fontsize=7, color='darkgreen')

ax.set_xlabel('x position (m)')
ax.set_ylabel('y position (m)')
ax.set_title('2D Kalman Filter: Trajectory with 2σ Uncertainty Ellipses')
ax.legend()
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('2d_kalman_ellipses.png', dpi=120, bbox_inches='tight')
plt.show()
print("Uncertainty ellipse plot saved.")

In [ ]:
# ── Demonstration with real-world-like GPS data (irregular noise) ──────────────
# Simulate a vehicle making a right turn
np.random.seed(7)

dt_gps = 0.5   # 2 Hz GPS
T_gps  = 100

A2, G2, H2, Sp2, Sm2 = build_system_matrices(dt_gps, sigma_a=1.0, sigma_m=8.0)

# True state: straight then turning
x_gps = np.zeros((T_gps, 4))
x_gps[0] = [0, 0, 10, 0]  # 10 m/s east

for k in range(1, T_gps):
    turn = 0.0
    if 30 <= k < 60:     # turning phase: add centripetal acceleration
        turn = -0.15     # turn right
    vx, vy = x_gps[k-1, 2], x_gps[k-1, 3]
    speed = np.sqrt(vx**2 + vy**2)
    ax_true = -turn * vy
    ay_true =  turn * vx
    w = np.array([ax_true, ay_true]) + np.random.multivariate_normal([0,0], 0.5**2*np.eye(2))
    x_gps[k] = A2 @ x_gps[k-1] + G2 @ w

y_gps = (H2 @ x_gps.T).T + np.random.multivariate_normal([0,0], 8.0**2*np.eye(2), T_gps)

m0_gps = np.array([y_gps[0,0], y_gps[0,1], 5.0, 0.0])
P0_gps = np.diag([10.**2, 10.**2, 5.**2, 5.**2])

mp_gps, Pp_gps, mf_gps, Pf_gps, _ = kalman_filter_2d(
    y_gps, A2, G2, H2, Sp2, Sm2, m0_gps, P0_gps)

fig, ax = plt.subplots(figsize=(10, 7))
ax.plot(x_gps[:, 0], x_gps[:, 1], 'k-', lw=2, label='True path')
ax.scatter(y_gps[:, 0], y_gps[:, 1], c='red', s=18, alpha=0.5, label='GPS (σ=8m)')
ax.plot(mf_gps[:, 0], mf_gps[:, 1], 'g-', lw=2.5, label='Kalman filter')

for k in range(0, T_gps, 15):
    ell = covariance_ellipse(mf_gps[k, :2], Pf_gps[k, :2, :2], n_std=2,
                              edgecolor='green', facecolor='green', alpha=0.1, lw=1.5)
    ax.add_patch(ell)

ax.set_xlabel('Easting (m)')
ax.set_ylabel('Northing (m)')
ax.set_title('GPS Filtering: Vehicle Making a Right Turn')
ax.legend()
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('2d_gps_turn.png', dpi=120, bbox_inches='tight')
plt.show()

rms_raw = np.sqrt(np.mean((y_gps - x_gps[:,:2])**2))
rms_kf  = np.sqrt(np.mean((mf_gps[:,:2] - x_gps[:,:2])**2))
print(f"RMS error — Raw GPS: {rms_raw:.2f} m  |  Kalman: {rms_kf:.2f} m")
print(f"Kalman reduces GPS error by {(1 - rms_kf/rms_raw)*100:.1f}%")

---
## Summary

| Question | Key Results |
|---|---|
| Q1 | Full derivation of Kalman predict/update from Gaussian conditioning |
| Q2 | Scalar Kalman filter with animated prior/posterior Gaussians |
| Q3A | Matrices $A, G, H$ from constant-velocity kinematics |
| Q3B | 2D Kalman filter with GPS noise reduction, uncertainty ellipses, velocity recovery |